In [1]:
from joblib import Parallel, delayed
from tqdm.notebook import tqdm

import sys
import pickle
import numpy as np
import pandas as pd
import networkx as nx
import scipy
from itertools import combinations

In [2]:
sys.path.insert(0, "src")

from grn import grn

np.random.seed(5)


In [3]:
all_results = []

for network_seed in range(10):
    np.random.seed(network_seed)

    #manually adjust GRN hyperparameters in an attempt to build GRN with more non-additive effects
    G = grn().add_structure(
        n=256,
        k=1,
        alpha=1e-99,
        beta=0.99,       # 1 - 1/r
        gamma=0.01,      # 1/r
        kappa=10,        # --w
        delta_in=100,
        delta_out=100,
    )

    G.simulate_rna(
        s=1e-4,
        dt=1e-2,
        tmax=20000,
        burnin=5000,
        tol=1e-3,
        save=True,
    )

    x_wt = np.zeros(256)
    y_wt = G.rna

    single_ko_expression = np.zeros((G.n, G.n))
    single_ko_logfc = np.zeros((G.n, G.n))
    single_ko_convergence = {}

    for gene in tqdm(range(G.n), desc=f"Network {network_seed}: single-gene KOs"):

        result = G.ko_nodes(
            genes=[gene],
            stats=("new_rna", "logfc", "convergence"),
            s=1e-4,
            dt=1e-2,
            tmax=20000,
            burnin=5000,
            tol=1e-3,
        )

        single_ko_expression[gene, :] = result["new_rna"].flatten()
        single_ko_logfc[gene, :] = result["logfc"].flatten()
        single_ko_convergence[gene] = result["convergence"]

    x_single_ko = np.eye(G.n)
    y_single_ko = single_ko_expression

    pairs = list(combinations(range(G.n), 2))

    def run_double_ko(pair):
        a, b = pair

        result = G.ko_nodes(
            genes=[a, b],
            stats=("new_rna", "logfc", "convergence"),
            s=0,
            dt=1e-2,
            tmax=20000,
            burnin=5000,
            tol=1e-3,
        )

        return (
            np.asarray(result["new_rna"]).flatten(),
            np.asarray(result["logfc"]).flatten(),
            np.asarray(result["convergence"]),
        )


    results = Parallel(
        n_jobs=64,
        pre_dispatch="n_jobs",
        prefer="processes",
        verbose=10,
    )(
        delayed(run_double_ko)(pair)
        for pair in pairs
    )

    double_expression = np.asarray(
        [r[0] for r in results],
        dtype=np.float32
    )

    double_logfc = np.asarray(
        [r[1] for r in results],
        dtype=np.float32
    )

    double_convergence = np.asarray(
        [r[2] for r in results],
        dtype=np.float32
    )


    #now prepare as training data

    pairs = np.array(
        pairs,
        dtype=np.int32
    )

    x_double_ko = np.zeros((pairs.shape[0], G.n))

    #now fill x for doubleKO with 1s

    rows = np.arange(pairs.shape[0])
    x_double_ko[rows, pairs[:, 0]] = 1.0
    x_double_ko[rows, pairs[:, 1]] = 1.0

    y_double_ko = double_expression


    # ============================================================
    # 10 TRAINING RUNS ON THIS GRN
    # ============================================================

    for training_seed in range(10):

        import torch
        from torch import nn

        torch.manual_seed(training_seed)
        device = torch.device("cpu")

        #for more precision
        #torch.backends.cuda.matmul.allow_tf32 = False
        #torch.backends.cudnn.allow_tf32 = False


        #model implementation without dropout
        model = nn.Sequential(
            nn.Linear(256, 512),
            nn.GELU(),
            nn.Linear(512, 512),
            nn.GELU(),
            nn.Linear(512, 256)
        ).to(device)

        linear = nn.Linear(256, 256).to(device)

        #implement AdamW
        import torch.optim as optim

        optimizer = optim.AdamW(
            model.parameters(),
            lr=1e-3,
            weight_decay=1e-2
        )


        #process data, standardise, put them on GPU

        rng = np.random.default_rng(training_seed)

        n_pairs = pairs.shape[0]
        nt, nv = int(0.5 * n_pairs), int(0.1 * n_pairs)

        indices = rng.permutation(n_pairs)
        test_idx = indices[:nt]
        val_idx = indices[nt:nt + nv]
        train_idx = indices[nt + nv:]

        xtr = np.concatenate(
            [x_wt[None, :], x_single_ko, x_double_ko[train_idx]],
            axis=0
        )

        ytr = np.concatenate(
            [y_wt[None, :], y_single_ko, y_double_ko[train_idx]],
            axis=0
        )

        xv = x_double_ko[val_idx]
        yv = y_double_ko[val_idx]

        x_test = x_double_ko[test_idx]
        y_test = y_double_ko[test_idx]

        mean, scale = (
            np.mean(ytr, axis=0),
            np.maximum(ytr.std(axis=0), 1e-6)
        )

        s_ytr = (ytr - mean) / scale

        xtr_t = torch.as_tensor(
            xtr,
            dtype=torch.float32,
            device=device
        )

        s_ytr_t = torch.as_tensor(
            s_ytr,
            dtype=torch.float32,
            device=device
        )

        xv_t = torch.as_tensor(
            xv,
            dtype=torch.float32,
            device=device
        )

        yv_t = torch.as_tensor(
            yv,
            dtype=torch.float32,
            device=device
        )

        x_test_t = torch.as_tensor(
            x_test,
            dtype=torch.float32,
            device=device
        )

        y_test_t = torch.as_tensor(
            y_test,
            dtype=torch.float32,
            device=device
        )

        mean_t = torch.as_tensor(
            mean,
            dtype=torch.float32,
            device=device
        )

        scale_t = torch.as_tensor(
            scale,
            dtype=torch.float32,
            device=device
        )


        import time
        import copy

        max_epochs = 3000
        batch_size = 256
        lr = 1e-3
        patience = 40
        min_lr = 1e-5
        lr_patience = 12

        best, best_epoch, stale, lr_stale = np.inf, 0, 0, 0
        history = []
        started = time.perf_counter()
        best_model_weights = None


        for epoch in range(1, max_epochs + 1):

            model.train()

            permutation = torch.randperm(
                xtr_t.shape[0],
                device=device
            )

            total = torch.zeros((), device=device)

            for start in range(
                0,
                xtr_t.shape[0],
                batch_size
            ):

                ii = permutation[start:(start + batch_size)]

                optimizer.zero_grad(set_to_none=True)

                loss = torch.mean(
                    (model(xtr_t[ii]) - s_ytr_t[ii]) ** 2
                )

                loss.backward()
                optimizer.step()

                total += loss.detach() * len(ii)

            train_loss = float(
                total.item() / xtr_t.shape[0]
            )

            model.eval()

            with torch.inference_mode():
                val_scaled = torch.cat(
                    [
                        model(chunk)
                        for chunk in xv_t.split(1024)
                    ]
                ).cpu().numpy()

            val_pred = (
                val_scaled.astype(np.float32)
                * scale
                + mean
            )

            raw_mse = float(
                np.mean((val_pred - yv) ** 2)
            )

            if (
                not np.isfinite(train_loss)
                or not np.isfinite(raw_mse)
            ):
                raise FloatingPointError(
                    f"Non-finite loss at epoch {epoch}"
                )

            history.append({
                "epoch": epoch,
                "train_scaled_mse": train_loss,
                "val_scaled_mse": float(
                    np.mean(
                        ((val_pred - yv) / scale) ** 2
                    )
                ),
                "val_raw_mse": raw_mse,
                "learning_rate": lr,
                "elapsed_seconds":
                    time.perf_counter() - started
            })

            if raw_mse < best:

                best = raw_mse
                best_epoch = epoch
                stale = 0
                lr_stale = 0

                best_model_weights = copy.deepcopy(
                    model.state_dict()
                )

            else:

                stale += 1
                lr_stale += 1

            if epoch == 1 or epoch % 10 == 0:

                print(
                    f"network {network_seed}, "
                    f"training {training_seed}, "
                    f"epoch {epoch}: "
                    f"train scaled MSE {train_loss:.6g}; "
                    f"val RMSE {raw_mse**.5:.7g}; "
                    f"lr {lr:g}; "
                    f"{time.perf_counter()-started:.0f}s",
                    flush=True
                )

            if (
                lr_stale >= lr_patience
                and lr > min_lr
            ):

                lr = max(
                    min_lr,
                    lr * .5
                )

                for group in optimizer.param_groups:
                    group["lr"] = lr

                lr_stale = 0

            if stale >= patience:
                break


        if best_model_weights is not None:
            model.load_state_dict(
                best_model_weights
            )


        from sklearn.linear_model import LinearRegression

        # 1. Initialize standard Linear Regression (no alpha/penalty)
        linear_model = LinearRegression(
            fit_intercept=True
        )

        # 2. Fit directly on the NumPy arrays
        linear_model.fit(
            xtr.astype(np.float64),
            s_ytr.astype(np.float64)
        )

        # 3. Predict and reverse normalization
        val_scaled = linear_model.predict(
            xv.astype(np.float64)
        )

        val_pred = (
            val_scaled.astype(np.float32)
            * scale
            + mean
        )

        # 4. Calculate RMSE
        raw_mse = float(
            np.mean(
                (val_pred - yv) ** 2
            )
        )

        print(
            f"Standard Linear Baseline Validation RMSE: "
            f"{raw_mse**.5:.7g}"
        )


        # 1. Generate predictions for the Test set
        model.eval()

        with torch.inference_mode():

            mlp_test_scaled = torch.cat(
                [
                    model(chunk)
                    for chunk
                    in x_test_t.split(1024)
                ]
            ).cpu().numpy()

        mlp_pred = (
            mlp_test_scaled.astype(np.float32)
            * scale
            + mean
        )


        # Retrieve linear model predictions
        linear_test_scaled = linear_model.predict(
            x_test.astype(np.float64)
        )

        linear_pred = (
            linear_test_scaled.astype(np.float32)
            * scale
            + mean
        )


        # 2. Reconstruct test pairs and baseline predictions
        test_pairs = pairs[test_idx]

        # The additive baseline is Single A + Single B - WT
        additive = (
            y_single_ko[test_pairs[:, 0]]
            + y_single_ko[test_pairs[:, 1]]
            - y_wt
        )

        y_true = y_test

        true_r = y_true - additive

        add_pair_mse = np.mean(
            true_r ** 2,
            axis=1
        )

        add_mse = float(
            np.mean(add_pair_mse)
        )


        # Helper function for Pearson correlation
        def correlation(a, b):

            a = np.asarray(a).ravel()
            b = np.asarray(b).ravel()

            if (
                np.std(a) < 1e-15
                or np.std(b) < 1e-15
            ):
                return None

            return float(
                np.corrcoef(a, b)[0, 1]
            )


        predictions = {
            "Additive Baseline": additive,
            "Linear Regression": linear_pred,
            "MLP": mlp_pred
        }

        metrics = []
        pair_losses = {}


        # 3. Calculate core metrics
        for name, pred in predictions.items():

            e2 = (pred - y_true) ** 2

            pair_losses[name] = e2.mean(axis=1)

            implied_residual = pred - additive

            mse = float(
                np.mean(e2)
            )

            pearson_corr = correlation(
                implied_residual,
                true_r
            )

            metrics.append({
                "Model": name,
                "MSE": mse,
                "RMSE": mse ** 0.5,
                "MSE Improvement vs Additive (%)":
                    100 * (1 - mse / add_mse),
                "Pearson (Implied vs True Residual)":
                    pearson_corr if pearson_corr else 0.0,
                "Fraction Pairs < Additive MSE":
                    float(
                        np.mean(
                            pair_losses[name]
                            < add_pair_mse
                        )
                    )
            })


        df = pd.DataFrame(
            metrics
        ).set_index("Model")


        # 4. Bootstrap 95% Confidence Intervals
        # for MSE Improvement (2000 resamples)

        rng = np.random.default_rng(9001)

        boot = {
            "Linear Regression": [],
            "MLP": []
        }

        for _ in range(2000):

            ii = rng.integers(
                0,
                len(test_pairs),
                size=len(test_pairs)
            )

            denom = add_pair_mse[ii].mean()

            for name in boot:

                boot[name].append(
                    100 * (
                        1
                        - pair_losses[name][ii].mean()
                        / denom
                    )
                )


        # Calculate and format CIs
        for name in boot:

            ci_lower, ci_upper = np.quantile(
                boot[name],
                [0.025, 0.975]
            )

            df.loc[
                name,
                "MSE Improv. 95% CI"
            ] = (
                f"[{ci_lower:.2f}%, "
                f"{ci_upper:.2f}%]"
            )


        df.loc[
            "Additive Baseline",
            "MSE Improv. 95% CI"
        ] = "-"


        # 5. Calculate Pearson Correlation
        # Improvement (MLP vs Linear)

        mlp_pearson = df.loc[
            "MLP",
            "Pearson (Implied vs True Residual)"
        ]

        lin_pearson = df.loc[
            "Linear Regression",
            "Pearson (Implied vs True Residual)"
        ]


        if lin_pearson != 0:

            pearson_improvement = (
                (mlp_pearson - lin_pearson)
                / abs(lin_pearson)
            ) * 100

            df.loc[
                "MLP",
                "Pearson Improv. vs Linear (%)"
            ] = pearson_improvement


        df.loc[
            "Linear Regression",
            "Pearson Improv. vs Linear (%)"
        ] = 0.0


        # Use np.nan instead of a string
        # to keep the column numerically typed
        df.loc[
            "Additive Baseline",
            "Pearson Improv. vs Linear (%)"
        ] = np.nan


        # ========================================================
        # SAVE RESULTS FROM THIS NETWORK/TRAINING RUN
        # ========================================================

        run_df = df.reset_index()

        run_df.insert(
            0,
            "Training Seed",
            training_seed
        )

        run_df.insert(
            0,
            "Network Seed",
            network_seed
        )

        all_results.append(run_df)


        # Full results accumulated so far
        all_results_df = pd.concat(
            all_results,
            ignore_index=True
        )

        # Save after every training run so that completed
        # runs are not lost if a later run fails.
        all_results_df.to_csv(
            "../grn_training_results.csv",
            index=False
        )


        # Compact one-row-per-run RMSE table
        # MLP vs regression vs additive
        rmse_by_run = all_results_df.pivot(
            index=[
                "Network Seed",
                "Training Seed"
            ],
            columns="Model",
            values="RMSE"
        ).reset_index()

        rmse_by_run.columns.name = None

        rmse_by_run.to_csv(
            "../grn_training_rmse_by_run.csv",
            index=False
        )


        # Format the DataFrame for clean output
        display_cols = [
            "RMSE",
            "MSE Improvement vs Additive (%)",
            "MSE Improv. 95% CI",
            "Pearson (Implied vs True Residual)",
            "Pearson Improv. vs Linear (%)",
            "Fraction Pairs < Additive MSE"
        ]


        print(
            f"\nNetwork seed {network_seed}, "
            f"training seed {training_seed}"
        )

        display(
            df[
                display_cols
            ].round(4).fillna("-")
        )


# ================================================================
# FINAL RESULT TABLES
# ================================================================

# 10 networks x 10 training runs x 3 models = 300 rows
all_results_df = pd.concat(
    all_results,
    ignore_index=True
)

# 100 rows, one for each Network Seed / Training Seed combination
rmse_by_run = all_results_df.pivot(
    index=[
        "Network Seed",
        "Training Seed"
    ],
    columns="Model",
    values="RMSE"
).reset_index()

rmse_by_run.columns.name = None

display(all_results_df)
display(rmse_by_run)

Network 0: single-gene KOs:   0%|          | 0/256 [00:00<?, ?it/s]

[Parallel(n_jobs=64)]: Using backend LokyBackend with 64 concurrent workers.
[Parallel(n_jobs=64)]: Done   8 tasks      | elapsed:   30.7s
[Parallel(n_jobs=64)]: Done  21 tasks      | elapsed:   31.0s
[Parallel(n_jobs=64)]: Done  34 tasks      | elapsed:   31.2s
[Parallel(n_jobs=64)]: Done  49 tasks      | elapsed:   31.4s
[Parallel(n_jobs=64)]: Done  64 tasks      | elapsed:   31.6s
[Parallel(n_jobs=64)]: Done  81 tasks      | elapsed:   32.2s
[Parallel(n_jobs=64)]: Done  98 tasks      | elapsed:   32.8s
[Parallel(n_jobs=64)]: Done 117 tasks      | elapsed:   33.5s
[Parallel(n_jobs=64)]: Done 136 tasks      | elapsed:   34.0s
[Parallel(n_jobs=64)]: Done 157 tasks      | elapsed:   34.7s
[Parallel(n_jobs=64)]: Done 178 tasks      | elapsed:   35.5s
[Parallel(n_jobs=64)]: Done 201 tasks      | elapsed:   36.4s
[Parallel(n_jobs=64)]: Done 224 tasks      | elapsed:   37.2s
[Parallel(n_jobs=64)]: Done 249 tasks      | elapsed:   38.0s
[Parallel(n_jobs=64)]: Done 274 tasks      | elapsed:  

network 0, training 0, epoch 1: train scaled MSE 0.910925; val RMSE 0.1566212; lr 0.001; 1s
network 0, training 0, epoch 10: train scaled MSE 0.512312; val RMSE 0.1233028; lr 0.001; 2s
network 0, training 0, epoch 20: train scaled MSE 0.471266; val RMSE 0.1190871; lr 0.001; 5s
network 0, training 0, epoch 30: train scaled MSE 0.440797; val RMSE 0.1174908; lr 0.001; 7s
network 0, training 0, epoch 40: train scaled MSE 0.412341; val RMSE 0.1158606; lr 0.001; 10s
network 0, training 0, epoch 50: train scaled MSE 0.37279; val RMSE 0.1154633; lr 0.001; 12s
network 0, training 0, epoch 60: train scaled MSE 0.335953; val RMSE 0.1158411; lr 0.0005; 14s
network 0, training 0, epoch 70: train scaled MSE 0.308009; val RMSE 0.1161971; lr 0.0005; 17s
network 0, training 0, epoch 80: train scaled MSE 0.291441; val RMSE 0.1161266; lr 0.00025; 19s
Standard Linear Baseline Validation RMSE: 0.125638

Network seed 0, training seed 0


,RMSE,MSE Improvement vs Additive (%),MSE Improv. 95% CI,Pearson (Implied vs True Residual),Pearson Improv. vs Linear (%),Fraction Pairs < Additive MSE
Model,,,,,,
Additive Baseline,0.1917,0.0000,-,0.0000,-,0.0000
Linear Regression,0.1256,57.0504,"[56.42%, 57.62%]",0.7552,0.0,0.8392
MLP,0.1154,63.7380,"[63.12%, 64.32%]",0.7985,5.7384,0.8937


network 0, training 1, epoch 1: train scaled MSE 0.90285; val RMSE 0.1572696; lr 0.001; 0s
network 0, training 1, epoch 10: train scaled MSE 0.520094; val RMSE 0.1243996; lr 0.001; 2s
network 0, training 1, epoch 20: train scaled MSE 0.468722; val RMSE 0.1193994; lr 0.001; 5s
network 0, training 1, epoch 30: train scaled MSE 0.443484; val RMSE 0.1176323; lr 0.001; 7s
network 0, training 1, epoch 40: train scaled MSE 0.407752; val RMSE 0.1164973; lr 0.001; 9s
network 0, training 1, epoch 50: train scaled MSE 0.371861; val RMSE 0.1163059; lr 0.001; 12s
network 0, training 1, epoch 60: train scaled MSE 0.333813; val RMSE 0.1168671; lr 0.001; 14s
network 0, training 1, epoch 70: train scaled MSE 0.30399; val RMSE 0.1169034; lr 0.0005; 16s
network 0, training 1, epoch 80: train scaled MSE 0.286785; val RMSE 0.1167204; lr 0.00025; 18s
Standard Linear Baseline Validation RMSE: 0.1258372

Network seed 0, training seed 1


...

,RMSE,MSE Improvement vs Additive (%),MSE Improv. 95% CI,Pearson (Implied vs True Residual),Pearson Improv. vs Linear (%),Fraction Pairs < Additive MSE
Model,,,,,,
Additive Baseline,0.0781,-0.0000,-,0.0000,-,0.0000
Linear Regression,0.0690,22.0510,"[20.83%, 23.23%]",0.4699,0.0,0.1496
MLP,0.0576,45.7395,"[44.28%, 47.16%]",0.6784,44.3895,0.1552


network 9, training 9, epoch 1: train scaled MSE 0.89434; val RMSE 0.1065846; lr 0.001; 0s
network 9, training 9, epoch 10: train scaled MSE 0.395855; val RMSE 0.06225665; lr 0.001; 2s
network 9, training 9, epoch 20: train scaled MSE 0.34325; val RMSE 0.05710493; lr 0.001; 4s
network 9, training 9, epoch 30: train scaled MSE 0.29681; val RMSE 0.05631768; lr 0.001; 7s
network 9, training 9, epoch 40: train scaled MSE 0.245424; val RMSE 0.05601776; lr 0.001; 9s
network 9, training 9, epoch 50: train scaled MSE 0.194675; val RMSE 0.05716714; lr 0.001; 11s
network 9, training 9, epoch 60: train scaled MSE 0.167186; val RMSE 0.05647214; lr 0.0005; 13s
network 9, training 9, epoch 70: train scaled MSE 0.152903; val RMSE 0.0566687; lr 0.00025; 15s
network 9, training 9, epoch 80: train scaled MSE 0.144435; val RMSE 0.05681306; lr 0.000125; 17s
Standard Linear Baseline Validation RMSE: 0.06720096

Network seed 9, training seed 9


,RMSE,MSE Improvement vs Additive (%),MSE Improv. 95% CI,Pearson (Implied vs True Residual),Pearson Improv. vs Linear (%),Fraction Pairs < Additive MSE
Model,,,,,,
Additive Baseline,0.0782,0.0000,-,0.0000,-,0.0000
Linear Regression,0.0692,21.7867,"[20.51%, 22.95%]",0.4683,0.0,0.1507
MLP,0.0577,45.5312,"[44.04%, 47.08%]",0.6803,45.2637,0.1530


,Network Seed,Training Seed,Model,MSE,RMSE,MSE Improvement vs Additive (%),Pearson (Implied vs True Residual),Fraction Pairs < Additive MSE,MSE Improv. 95% CI,Pearson Improv. vs Linear (%)
0,0,0,Additive Baseline,0.036751,0.191705,0.000000e+00,0.000000,0.000000,-,NaN
1,0,0,Linear Regression,0.015784,0.125636,5.705039e+01,0.755163,0.839216,"[56.42%, 57.62%]",0.000000
2,0,0,MLP,0.013327,0.115441,6.373804e+01,0.798497,0.893689,"[63.12%, 64.32%]",5.738364
3,0,1,Additive Baseline,0.036887,0.192061,-2.220446e-14,0.000000,0.000000,-,NaN
4,0,1,Linear Regression,0.015889,0.126053,5.692465e+01,0.754277,0.841176,"[56.35%, 57.51%]",0.000000
...,...,...,...,...,...,...,...,...,...,...
295,9,8,Linear Regression,0.004759,0.068983,2.205101e+01,0.469852,0.149632,"[20.83%, 23.23%]",0.000000
296,9,8,MLP,0.003312,0.057554,4.573945e+01,0.678416,0.155208,"[44.28%, 47.16%]",44.389512
297,9,9,Additive Baseline,0.006120,0.078231,0.000000e+00,0.000000,0.000000,-,NaN
298,9,9,Linear Regression,0.004787,0.069186,2.178672e+01,0.468300,0.150674,"[20.51%, 22.95%]",0.000000


,Network Seed,Training Seed,Additive Baseline,Linear Regression,MLP
0,0,0,0.191705,0.125636,0.115441
1,0,1,0.192061,0.126053,0.115824
2,0,2,0.191055,0.125186,0.114999
3,0,3,0.191998,0.125410,0.115741
4,0,4,0.192487,0.126309,0.116125
...,...,...,...,...,...
95,9,5,0.078094,0.069176,0.057164
96,9,6,0.078148,0.069083,0.057940
97,9,7,0.078477,0.069020,0.057040
98,9,8,0.078133,0.068983,0.057554


In [4]:
mlp_ci_table = (
    all_results_df
    .query("Model == 'MLP'")
    .pivot(
        index="Training Seed",
        columns="Network Seed",
        values="MSE Improv. 95% CI"
    )
)

display(mlp_ci_table)

Network Seed,0,1,2,3,4,5,6,7,8,9
Training Seed,,,,,,,,,,
0,"[63.12%, 64.32%]","[47.07%, 49.33%]","[35.38%, 38.68%]","[49.54%, 51.80%]","[66.23%, 67.69%]","[50.98%, 53.95%]","[55.93%, 57.70%]","[45.98%, 48.39%]","[51.80%, 54.29%]","[44.56%, 47.55%]"
1,"[63.03%, 64.23%]","[45.42%, 48.07%]","[33.75%, 37.46%]","[50.34%, 52.68%]","[67.50%, 68.87%]","[51.98%, 54.92%]","[55.25%, 57.14%]","[46.68%, 48.99%]","[50.71%, 53.28%]","[45.24%, 48.19%]"
2,"[63.14%, 64.37%]","[46.31%, 48.61%]","[35.98%, 39.10%]","[50.85%, 53.01%]","[66.64%, 68.07%]","[50.15%, 53.08%]","[55.94%, 57.79%]","[46.02%, 48.38%]","[50.57%, 53.06%]","[45.31%, 48.34%]"
3,"[63.02%, 64.28%]","[47.54%, 49.96%]","[36.91%, 40.10%]","[51.01%, 53.27%]","[66.29%, 67.76%]","[49.29%, 52.24%]","[56.95%, 58.73%]","[45.46%, 47.82%]","[51.55%, 54.04%]","[43.49%, 46.54%]"
4,"[63.00%, 64.23%]","[47.07%, 49.41%]","[34.63%, 37.85%]","[51.24%, 53.39%]","[66.97%, 68.35%]","[50.94%, 53.90%]","[55.48%, 57.20%]","[45.62%, 48.06%]","[50.95%, 53.47%]","[43.86%, 46.99%]"
5,"[63.40%, 64.66%]","[46.26%, 48.51%]","[36.53%, 39.73%]","[50.20%, 52.52%]","[65.78%, 67.19%]","[50.77%, 53.59%]","[55.89%, 57.71%]","[46.86%, 49.19%]","[51.11%, 53.66%]","[45.04%, 47.85%]"
6,"[62.75%, 63.94%]","[46.39%, 48.62%]","[35.26%, 38.50%]","[50.18%, 52.37%]","[66.96%, 68.30%]","[50.46%, 53.36%]","[55.64%, 57.55%]","[45.83%, 48.25%]","[50.47%, 52.95%]","[43.48%, 46.61%]"
7,"[62.89%, 64.14%]","[46.66%, 48.96%]","[35.64%, 38.65%]","[50.41%, 52.66%]","[67.67%, 69.02%]","[50.64%, 53.61%]","[56.00%, 57.80%]","[45.18%, 47.61%]","[52.43%, 54.88%]","[45.78%, 48.50%]"
8,"[63.13%, 64.48%]","[46.51%, 48.89%]","[35.65%, 38.76%]","[50.17%, 52.32%]","[66.92%, 68.31%]","[50.78%, 53.63%]","[55.64%, 57.34%]","[46.05%, 48.38%]","[50.34%, 52.94%]","[44.28%, 47.16%]"


In [5]:
mlp_ci_table = (
    all_results_df
    .query("Model == 'MLP'")
    .pivot(
        index="Training Seed",
        columns="Network Seed",
        values="MSE Improvement vs Additive (%)"
    )
)

display(mlp_ci_table)

Network Seed,0,1,2,3,4,5,6,7,8,9
Training Seed,,,,,,,,,,
0,63.738039,48.195486,37.082449,50.671678,66.968011,52.505548,56.819271,47.177657,53.097677,46.098384
1,63.631960,46.712883,35.588884,51.548126,68.220579,53.472290,56.167880,47.798686,52.098199,46.779816
2,63.769560,47.502772,37.534678,51.951997,67.389886,51.676579,56.873694,47.202643,51.816142,46.856235
3,63.660335,48.809291,38.483727,52.152739,67.032673,50.763833,57.841400,46.635037,52.811882,45.017426
4,63.604561,48.254931,36.274671,52.308364,67.675746,52.414467,56.362183,46.867639,52.212202,45.457825
5,64.053065,47.399246,38.127699,51.401235,66.519778,52.177714,56.758096,48.040317,52.427761,46.419189
6,63.332327,47.514500,36.964514,51.327862,67.622612,51.942774,56.598601,47.072933,51.727057,45.030242
7,63.514853,47.856760,37.187371,51.536179,68.347829,52.173576,56.888611,46.440247,53.661398,47.170889
8,63.798886,47.738070,37.284791,51.257240,67.619033,52.231691,56.518703,47.215903,51.620881,45.739450
